In [2]:
# simulate_substitution.ipynb

import random
import time
import pandas as pd
import numpy as np
from substitution_extension import ClassificationDatabase, SalesDatabase, Product, Company, System
from SSELF_base import FootprintDatabase

# Step 1: Set up classification data
classification_data = [
    {"class_code": "HS 2517", "class_name": "Crushed stone", "function": "Fills space", "unit": "m3"},
    {"class_code": "HS 2905", "class_name": "Alcohols", "function": "Chemical feedstock", "unit": "kg"},
    {"class_code": "HS 2618", "class_name": "Slag", "function": "Binder", "unit": "kg"},
]
classification_db = ClassificationDatabase(classification_data)

# Step 2: Initialize sales database for last year
last_year_sales = SalesDatabase(2023, classification_db)
product_1 = Product(12, "Granite", "m3", "Company_11", "HS 2517", "primary", 0.5, classification_db)
product_2 = Product(14, "Glycerol", "kg", "Company_12", "HS 2905", "primary", 0.05, classification_db)
product_3 = Product(16, "Slag", "kg", "Company_13", "HS 2618", "primary", 0.7, classification_db)

last_year_sales.add_sales(product_1, 1000)
last_year_sales.add_sales(product_2, 2000)
last_year_sales.add_sales(product_3, 3000)

# Step 3: Add dummy entries to last_year_sales
codes = classification_db.data['class_code'].unique()
next_id = 17
extras = []
base_function_outputs = {
    "HS 2517": 0.5,
    "HS 2905": 0.05,
    "HS 2618": 0.7,
}

for code in codes:
    for _ in range(2):
        unit = classification_db.get_class_info(code)[2]
        extras.append({
            "id": next_id,
            "class_code": code,
            "sales_volume": np.random.uniform(500, 2000),
            "unit": unit,
            "function_output": base_function_outputs[code],  #  Use fixed value
        })
        next_id += 1
last_year_sales.data = pd.concat([last_year_sales.data, pd.DataFrame(extras)], ignore_index=True)

# Step 4: Build the system
system = System(num_companies=13, num_products=19, classification_db=classification_db)

# Add co-products manually to companies 11, 12, and 13
co_product_details = [
    ("Crushed granite", "HS 2517", 0.025),
    ("Glycerol", "HS 2905", 0.05),
    ("Blast Furnace Slag", "HS 2618", 0.1)
]
for i, (name, code, fout) in enumerate(co_product_details):
    company = system.companies[f"Company_{11 + i}"]
    new_id = 17 + i  # Continuing from next_id above
    co_product = Product(new_id, name, "kg", company, code, "secondary", fout, classification_db)
    company.add_product(co_product)
    if company.sales.empty:
        company.sales = pd.DataFrame(columns=["Sales"])
    company.sales.loc[new_id] = [100]

# Step 5: Create and initialize footprint databases
fp_2023 = FootprintDatabase(2023)
fp_2024 = FootprintDatabase(2024)

# Initialize 2023/2024 footprints for all actual products
for company in system.companies.values():
    for product in company.products:
        fp_2023.report(product.product_id, random.uniform(1, 100))
        fp_2024.report(product.product_id, 0)

# Also initialize dummy entries from last_year_sales to support substitution logic
for row in last_year_sales.data.itertuples():
    if row.id not in fp_2023.data["id"].values:
        fp_2023.report(row.id, random.uniform(1, 100))
# Step 6: Run simulation loop
updates_completed = False
iteration = 0
start_time = time.time()
company_names = list(system.companies.keys())

while not updates_completed:
    iteration += 1
    print(f"\n--- Iteration {iteration} ---")
    any_updated = False
    unchecked = company_names[:]

    while unchecked:
        cname = random.choice(unchecked)
        comp = system.companies[cname]
        if comp.check_update_needed(fp_2024, last_year_sales, fp_2023):
            print(f"{cname} was updated.")
            comp.update_footprint(fp_2024, fp_2023, last_year_sales)  # ← MISSING!
            any_updated = True

        unchecked.remove(cname)

    updates_completed = not any_updated

end_time = time.time()
print(f"\n Updates completed in {iteration} iterations.")
print(f" Time taken: {end_time - start_time:.2f} seconds")

# Final reporting
print("\n Final Footprints:")
for cname, comp in system.companies.items():
    print(f"{cname}: {comp.latest_update:.4f} kg CO2e/unit")

print("\n Update Count:")
total_updates = sum(c.num_updates for c in system.companies.values())
for cname, comp in system.companies.items():
    print(f"{cname}: {comp.num_updates} updates")
print(f"Total updates: {total_updates}")

print("\n Carbon balance check:")
for name, company in system.companies.items():
    company.check_carbon_balance(fp_2024)



--- Iteration 1 ---
  [DEBUG] Average footprint for HS 2517: total impact = 45749.8097, weight = 1295.7360

[Company_11] Checking product 11 (Product 11)
  Current footprint: 0.000000, Simulated footprint: -0.718850
  Difference: 0.718850
Company_11 was updated.
  [DEBUG] Average footprint for HS 2517: total impact = 45749.8097, weight = 1295.7360

[Company_2] Checking product 2 (Product 2)
  Current footprint: 0.000000, Simulated footprint: 19.652958
  Difference: 19.652958
Company_2 was updated.
  [DEBUG] Average footprint for HS 2905: total impact = 11879.1793, weight = 235.1843

[Company_12] Checking product 12 (Product 12)
  Current footprint: 0.000000, Simulated footprint: 1.738009
  Difference: 1.738009
Company_12 was updated.
  [DEBUG] Average footprint for HS 2905: total impact = 11879.1793, weight = 235.1843
  [DEBUG] Average footprint for HS 2618: total impact = 102859.3989, weight = 3396.5540

[Company_13] Checking product 13 (Product 13)
  Current footprint: 0.000000, Sim

In [ ]:
from substitution_extension import Company
print(Company.__module__)
print(dir(Company))

In [3]:
print(system.companies["Company_1"].__class__)
print(dir(system.companies["Company_1"]))


<class 'substitution_extension.Company'>
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'add_product', 'check_carbon_balance', 'check_update_needed', 'direct_impacts', 'get_average_footprint', 'latest_update', 'name', 'num_updates', 'products', 'purchases', 'report_footprint', 'sales', 'update_footprint', 'year']
